# 06. ANFIS Data Preparation

### Objective
Construct the 6-feature input dataset and the target multiplier for ANFIS/MLP training.

### I/O
- **Reads**: `../../data/processed/5_clustered_telemetry.csv`
- **Writes**: `../../data/processed/6_anfis_dataset.csv`
- **Writes**: `../../data/processed/6_anfis_teacher_dataset.csv`

In [1]:
import pandas as pd
import numpy as np

INPUT_PATH   = '../../data/processed/5_clustered_telemetry.csv'
OUTPUT_PATH  = '../../data/processed/6_anfis_dataset.csv'
TEACHER_PATH = '../../data/processed/6_anfis_teacher_dataset.csv'

print('Libraries imported')

Libraries imported


In [2]:
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df)} rows')

Loaded 3240 rows


The 6 ANFIS input features: 3 soft membership scores + 3 temporal deltas. These were determined through the feature ablation study in Experiment F. Experiment F is: `experiments/experiment_F_anfis_ablation/Experiment_F_Complete.ipynb`

The experiment ran a leave-one-out ablation across all 6 features and also compared soft-only (3 features) vs combined (6 features). The result was that removing any delta feature caused a significantly larger MAE increase than removing a soft feature. The full 6-feature set achieved the best balance, and dropping any single feature degraded accuracy measurably. That is why all 6 features are used below.

In [3]:
input_features = [
    'soft_combat',  'soft_collect',  'soft_explore',
    'delta_combat', 'delta_collect', 'delta_explore'
]

print('ANFIS input features (6):')
for i, f in enumerate(input_features, 1):
    print(f'  {i}. {f}')

ANFIS input features (6):
  1. soft_combat
  2. soft_collect
  3. soft_explore
  4. delta_combat
  5. delta_collect
  6. delta_explore


Target variable computation - Option B delta-weighted formula.

Experiment C was run to decide which target formulation to use for training. Experiment C is: `experiments/experiment_C_target_variable/Experiment_C_Complete.ipynb`

The experiment compared two options:
 - Option A (static): T = 1 + 0.22*combat_c + 0.18*collect_c + 0.15*explore_c - 0.25*death_rate
   This produced near-zero variance (sigma ~0.011) because soft scores sum to 1.0, so the target sits near a constant. The MLP trained on Option A collapsed to predicting the mean every time - R2 was near zero, the model learned nothing.
 - Option B (delta-weighted): adds 0.55*delta_combat + 0.40*delta_collect + 0.35*delta_explore
   on top of the static base. Deltas are signed and can swing large when a player switches archetype, pushing target variance up 5.5x (sigma ~0.062). The MLP trained on Option B achieved R2=0.926 on test data.

Because of that result, Option B is implemented below.

In [4]:
# Center soft memberships around zero so they contribute as bias signals
combat_c  = df['soft_combat']  - 0.5
collect_c = df['soft_collect'] - 0.5
explore_c = df['soft_explore'] - 0.5

delta_combat  = df['delta_combat'].fillna(0)
delta_collect = df['delta_collect'].fillna(0)
delta_explore = df['delta_explore'].fillna(0)

# Normalize death count to [0,1] using 95th percentile to avoid outlier deaths
# dominating the penalty term for long sessions with a few unlucky streaks.
deaths    = df['death_count'].fillna(0)
death_rate = deaths / (deaths.quantile(0.95) + 1.0)
death_rate = death_rate.clip(0, 1)

df['target_multiplier'] = (
      1
    + 0.22 * combat_c
    + 0.18 * collect_c
    + 0.15 * explore_c
    + 0.55 * delta_combat
    + 0.40 * delta_collect
    + 0.35 * delta_explore
    - 0.25 * death_rate
)

# Clamp to [0.6, 1.4] - the playable difficulty range agreed with the game design spec.
# Going below 0.6 (40% easier) would make the game trivially easy for struggling players.
# Going above 1.4 would make it punishing for already-skilled players.
df['target_multiplier'] = np.clip(df['target_multiplier'], 0.6, 1.4)

t = df['target_multiplier']
print('='*50)
print('Option B Target Statistics (FINAL)')
print('='*50)
print(f'Min:  {t.min():.4f}')
print(f'Max:  {t.max():.4f}')
print(f'Mean: {t.mean():.4f}')
print(f'Std:  {t.std():.4f}')
print(f'Span: {t.max()-t.min():.4f}')
print('='*50)

Option B Target Statistics (FINAL)
Min:  0.6000
Max:  1.1213
Mean: 0.9103
Std:  0.0760
Span: 0.5213


In [5]:
# Save the full ANFIS dataset (userId + timestamp kept for traceability)
final_cols = ['userId', 'timestamp', 'cluster'] + input_features + ['target_multiplier']
anfis_df = df[final_cols].copy()
anfis_df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')

Saved to ../../data/processed/6_anfis_dataset.csv


In [6]:
# Save an explicit teacher dataset with just inputs and the teacher signal.
# This makes the MLP training contract clear: the model learns to approximate
# what an ideal ANFIS would produce given the 6D input vector.
teacher_df = anfis_df[input_features].copy()
teacher_df['anfis_teacher_output'] = anfis_df['target_multiplier'].values
teacher_df.to_csv(TEACHER_PATH, index=False)
print(f'Saved teacher dataset to {TEACHER_PATH}')
print(f'  Rows: {len(teacher_df)}')
print(f'  Columns: {list(teacher_df.columns)}')
print(f'  Teacher range: [{teacher_df["anfis_teacher_output"].min():.4f}, {teacher_df["anfis_teacher_output"].max():.4f}]')

Saved teacher dataset to ../../data/processed/6_anfis_teacher_dataset.csv
  Rows: 3240
  Columns: ['soft_combat', 'soft_collect', 'soft_explore', 'delta_combat', 'delta_collect', 'delta_explore', 'anfis_teacher_output']
  Teacher range: [0.6000, 1.1213]
